# EVI exploring

Code for exploring the patterns of vegetation recovery to determined the timeseries.

Used Google Earth Enginge (script in repository) to compute yearly raster files continaing the mean annual EVI. 

With Landsat 8 OLI/TIRS 2013-present, we used data from 2013-2025 


55. [Patterns in vegetation recovery](#EVI-Landsat)

# EVI Landsat

Using the computed yearly raster files, we can visualize the data

In [ ]:
import numpy as np
from pathlib import Path
import rasterio
import rioxarray as rxr

Below i am creating a workflow assuming... for when the data is ready

Two functions are define:

Given an open raster, extract EVI at a point :
        
        def get_evi(src, lon, lat) 

Given a point, loop through years and collect EVI values:
        
        def get_evi_timeseries(lon,lat)



In [ ]:
def get_evi(src, lon, lat): #here the src is the open raster 

    row,col = src.index(lon, lat) # convert coordinates of the info pixel to row and column of a raster. 

    #where are the pixels:
    win= rasterio.windows.Wuindows(col - buffer_px, row - buffer_px, buffer_px*2+1, buffer_px*2+1) 

    #read 1 band of the tif, only within that determined window:
    data = src.read(1,window= win).astype(float)

    #ignore NaN
    #use src.nodata which are no data pixels, [] reeplaces the true (false)
    data[data == src.nodata] = np.nan

    #Average the values that fall within that window, and ommits those NaN 
    return np.nanmean(data)

In [ ]:
EVI_DIR = Path('./EVI_per_year_data') # update the path folder 
YEARS = list(range(2013, 2025))
fire_year = 2015
buffer_px = 2 # read 2 pixels in each direction 

DESCRIPTION = 'summatra' # update the description of the data

#Sub sample of points to visualize their patterns
points = {
     # point_0(lon, lat)
    'point_1' : (105.00966314032637, -3.5518049355658503),
}
# making a diccionary with the interest points 
# or we can select them randomm??


In [ ]:
def get_evi_timeseries(lon, lat):
    results = {}
    for year in YEARS:
        tif = EVI_DIR / f'EVI_{DESCRIPTION}_{year}.tif'
        if not tif.exists():
            print(f'{year}: not found')
            continue
        with rasterio.open(tif) as src:
            evi = get_evi(src, lon, lat)   # <-- calls function 1
            results[year] = evi
            print(f'{year}: EVI = {evi:.4f}')
    return results
